In [1]:
import pandas as pd
from pathlib import Path


In [2]:
data_2010_star = pd.read_csv("../data/output/data-2010-star.csv")
data_2011_star = pd.read_csv("../data/output/data-2011-star.csv")
data_2012_star = pd.read_csv("../data/output/data-2012-star.csv")
data_2013_star = pd.read_csv("../data/output/data-2013-star.csv")
data_2014_star = pd.read_csv("../data/output/data-2014-star.csv")
data_2015_star = pd.read_csv("../data/output/data-2015-star.csv")




/tmp/ipykernel_3370090/3603335967.py:3: DtypeWarning: Columns (63) have mixed types. Specify dtype option on import or set low_memory=False.
  data_2012_star = pd.read_csv("../data/output/data-2012-star.csv")


In [3]:
for name, df in [
    ("2010", data_2010_star),
    ("2011", data_2011_star),
    ("2012", data_2012_star),
    ("2013", data_2013_star),
    ("2014", data_2014_star),
    ("2015", data_2015_star),
]:
    print(name, len(df.columns))


2010 111
2011 114
2012 115
2013 114
2014 113
2015 113


In [4]:
print(set(data_2010_star.columns) - set(data_2015_star.columns))
print(set(data_2015_star.columns) - set(data_2010_star.columns))


set()
{'org_parent', 'metric_33'}


In [5]:
all_years = [
    data_2010_star,
    data_2011_star,
    data_2012_star,
    data_2013_star,
    data_2014_star,
    data_2015_star
]

final_star= pd.concat(all_years, ignore_index=True)


In [6]:
import numpy as np  # needed for np.where in market_share

summary = (
    final_star.loc[final_star["year"].between(2010, 2015)]
        .assign(
            star_rating=lambda d: pd.to_numeric(d["partcd_score"], errors="coerce"),
            enrollment=lambda d: pd.to_numeric(d["avg_enrollment"], errors="coerce"),
            market_share=lambda d: np.where(
                pd.to_numeric(d["avg_enrolled"], errors="coerce") > 0,
                pd.to_numeric(d["avg_enrollment"], errors="coerce") / pd.to_numeric(d["avg_enrolled"], errors="coerce"),
                np.nan
            ),
            rated=lambda d: pd.to_numeric(d["partcd_score"], errors="coerce").notna()
        )
        .groupby("year")
        .agg(
            mean_star_rating=("star_rating", "mean"),
            mean_enrollments=("enrollment", "mean"),
            mean_market_share=("market_share", "mean"),
            n_plans=("planid", "count"),
            n_rated=("rated", "sum"),
        )
        .reset_index()
        .sort_values("year")
)

summary

,year,mean_star_rating,mean_enrollments,mean_market_share,n_plans,n_rated
0,2010,2.969662,256.353663,0.064564,108222,59579
1,2011,3.165988,343.358734,0.085547,68003,50808
2,2012,3.352218,377.336484,0.085568,67254,58177
3,2013,3.559948,395.516174,0.081451,68117,64456
4,2014,3.824169,436.637121,0.079989,62344,58636
5,2015,3.977247,469.679258,0.078689,65507,60169


In [7]:
print(final_star.shape)
print(final_star['year'].value_counts().sort_index())


(439447, 115)
year
2010    108222
2011     68003
2012     67254
2013     68117
2014     62344
2015     65507
Name: count, dtype: int64


In [8]:
output_path = "../data/output/final_star.csv"
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
final_star.to_csv(output_path, index=False)
print(f"Data saved to {output_path}")


Data saved to ../data/output/final_star.csv
